In [3]:
import pandas as pd
import os

# ================= 配置区 =================
KG_FILE = "kg.csv"  # PrimeKG 原始文件路径

# 针对 ADNI 数据集深度定制的扩充侦察组
SEARCH_GROUPS = {
    "【核心-AD与认知(含APOE)】": [
        "Alzheimer", "Dementia", "Memory", "Cognitive", "Amyloid", "Tau", "Neurofibrillary",
        "Apolipoprotein" # 抓取 APOE 相关生物通路
    ],
    "【ADNI-心脑血管】": [
        "Hypertension", "Cardiovascular", "Heart Attack", "Stroke", "Cerebrovascular", 
        "Myocardial infarction", "Ischemia"
    ],
    "【ADNI-精神与神经症状(NPI-Q)】": [
        "Psychiatric", "Depression", "Depressive", "Anxiety", "Agitation", "Apathy", 
        "Delusion", "Hallucination", "Euphoria", "Irritability", "Disinhibition",
        "Sleep", "Insomnia", "Appetite"
    ],
    "【ADNI-生活习惯与其他(吸烟/饮酒)】": [
        "Alcohol", "Ethanol", "Smoking", "Tobacco", "Nicotine"
    ]
}

def main():
    print(f"🚀 [Step 1] 正在加载 {KG_FILE} 进行 ADNI 适配侦察(扩充版)...")
    
    try:
        # 只读取必要列以节省内存
        df = pd.read_csv(KG_FILE, usecols=['relation', 'x_name', 'x_type', 'y_name', 'y_type'], low_memory=False)
    except ValueError:
        df = pd.read_csv(KG_FILE, low_memory=False)
        
    print(f"✅ PrimeKG 加载完成，共 {len(df)} 条数据。\n")

    # 1. 关键词命中测试
    print("="*60)
    print("🔎 1. 关键词命中测试 (确认 ADNI 特征能否映射到生物图谱)")
    print("="*60)
    
    # 剔除药物和暴露 (我们要找的是生物学机制)
    mask_bio = (~df['x_type'].isin(['drug', 'exposure'])) & (~df['y_type'].isin(['drug', 'exposure']))
    df_bio = df[mask_bio]

    found_keywords = set()

    for group, keywords in SEARCH_GROUPS.items():
        print(f"\n--- {group} ---")
        pattern = '|'.join(keywords)
        
        # 模糊匹配
        mask = df_bio['x_name'].astype(str).str.contains(pattern, case=False, na=False) | \
               df_bio['y_name'].astype(str).str.contains(pattern, case=False, na=False)
        subset = df_bio[mask]
        
        if len(subset) > 0:
            print(f"   ✅ 命中 {len(subset)} 条相关关系")
            related_nodes = set(subset['x_name']) | set(subset['y_name'])
            hits = [n for n in related_nodes if any(k.lower() in str(n).lower() for k in keywords)]
            print(f"   📌 典型节点示例: {list(hits)[:5]}")
            found_keywords.update(keywords)
        else:
            print(f"   ❌ 警告：未找到相关节点！请检查拼写或同义词。")

    # 2. 黑名单侦察 (Hub Node Detection)
    print("\n" + "="*60)
    print("💣 2. 黑名单侦察 (找出连接数过高且无特异性的干扰节点)")
    print("="*60)
    print("   说明：以下节点如果在 Top 20 且是通用解剖部位(如 kidney, lung)或通用概念，")
    print("   务必添加到下一步的 BLACKLIST_NODES 中！\n")

    # 统计所有节点的度 (Degree)
    all_nodes = pd.concat([df_bio['x_name'], df_bio['y_name']])
    node_counts = all_nodes.value_counts().head(30)

    print(f"{'Rank':<5} | {'Node Name':<40} | {'Count':<10}")
    print("-" * 60)
    for i, (node, count) in enumerate(node_counts.items()):
        print(f"{i+1:<5} | {node:<40} | {count:<10}")

    print("\n✅ [Step 1] 侦察完成！请根据上述结果：")
    print("   1. 确认 KEYWORDS 列表是否需要增删。")
    print("   2. 将高频干扰节点 (如 'Blood', 'Cytoplasm' 等) 复制到下一步的 BLACKLIST_NODES。")

if __name__ == "__main__":
    if os.path.exists(KG_FILE):
        main()
    else:
        print(f"❌ 找不到文件: {KG_FILE}")

🚀 [Step 1] 正在加载 kg.csv 进行 ADNI 适配侦察(扩充版)...
✅ PrimeKG 加载完成，共 8100498 条数据。

🔎 1. 关键词命中测试 (确认 ADNI 特征能否映射到生物图谱)

--- 【核心-AD与认知(含APOE)】 ---
   ✅ 命中 8248 条相关关系
   📌 典型节点示例: ['amyloid-beta clearance by transcytosis', 'ataxia with dementia', 'positive regulation of apolipoprotein binding', 'innate immunity memory response', 'adaptive immune memory response involving T cells and B cells']

--- 【ADNI-心脑血管】 ---
   ✅ 命中 3864 条相关关系
   📌 典型节点示例: ['pediatric arterial ischemic stroke', 'renovascular hypertension (disease)', 'Ischemic stroke', 'pulmonary hypertension, primary', 'autosomal recessive leukoencephalopathy-ischemic stroke-retinitis pigmentosa syndrome']

--- 【ADNI-精神与神经症状(NPI-Q)】 ---
   ✅ 命中 6008 条相关关系
   📌 典型节点示例: ['foveal depression', 'continuous spikes and waves during sleep', 'Agitation', 'autoimmune encephalopathy with parasomnia and obstructive sleep apnea', 'sleep-wake disorder']

--- 【ADNI-生活习惯与其他(吸烟/饮酒)】 ---
   ✅ 命中 3272 条相关关系
   📌 典型节点示例: ['phosphatidyl-N-methylethanolamine N-me

In [4]:
import pandas as pd
import os

# ADNI 文件列表
ADNI_FILES = ['AD.csv', 'mci.csv', 'normal.csv']

# 病史字段 (适配 ADNI)
HIS_COLS = {
    "Heart Attack (心梗/心血管)": "his_CVHATT",
    "Hypertension (高血压)": "his_HYPERTEN",
    "Psychiatric Dis. (精神疾病)": "his_PSYCDIS", 
    "Alcohol Abuse (酒精滥用)": "his_Alcohol",
    "Other Depression (其他抑郁)": "his_DEPOTHR",
    "Stroke (中风)": "his_CBSTROKE"
}

def analyze_adni_prevalence():
    print("📊 正在统计 ADNI 临床数据中的病史出现频率...\n")
    
    total_patients = 0
    stats = {k: 0 for k in HIS_COLS.keys()}
    
    for fname in ADNI_FILES:
        if not os.path.exists(fname):
            continue
            
        try:
            df = pd.read_csv(fname)
            total_patients += len(df)
            
            for label, col in HIS_COLS.items():
                if col in df.columns:
                    # ADNI 编码通常也是: 1=Yes, 0=No
                    # 我们只统计明确为 1 (Yes) 的
                    count = df[col].apply(lambda x: 1 if x == 1 else 0).sum()
                    stats[label] += count
        except Exception as e:
            print(f"⚠️ 读取 {fname} 出错: {e}")

    print(f"👥 总样本量: {total_patients}")
    print("-" * 50)
    print(f"{'特征':<25} | {'患病人数':<10} | {'占比 (%)':<10}")
    print("-" * 50)
    
    # 排序并打印
    sorted_stats = sorted(stats.items(), key=lambda x: x[1], reverse=True)
    
    for label, count in sorted_stats:
        ratio = (count / total_patients) * 100 if total_patients > 0 else 0
        print(f"{label:<25} | {count:<10} | {ratio:.1f}%")
        
    print("-" * 50)
    print("💡 决策建议：")
    print("   - 占比 < 5% 的特征，建议从知识图谱中【完全剔除】，因为样本太少无法训练。")
    print("   - 占比 > 20% 的特征，是核心驱动力，必须保留其复杂的生物学机制。")

# 执行分析
analyze_adni_prevalence()

📊 正在统计 ADNI 临床数据中的病史出现频率...

👥 总样本量: 837
--------------------------------------------------
特征                        | 患病人数       | 占比 (%)    
--------------------------------------------------
Heart Attack (心梗/心血管)     | 584        | 69.8%
Hypertension (高血压)        | 415        | 49.6%
Psychiatric Dis. (精神疾病)   | 268        | 32.0%
Alcohol Abuse (酒精滥用)      | 31         | 3.7%
Other Depression (其他抑郁)   | 10         | 1.2%
Stroke (中风)               | 0          | 0.0%
--------------------------------------------------
💡 决策建议：
   - 占比 < 5% 的特征，建议从知识图谱中【完全剔除】，因为样本太少无法训练。
   - 占比 > 20% 的特征，是核心驱动力，必须保留其复杂的生物学机制。
